# Crypto Price Prediction

This notebook walks through the complete pipeline:
1. **Data Collection** — Fetch historical OHLCV data from CoinGecko
2. **Exploratory Data Analysis** — Visualize price history and patterns
3. **Feature Engineering** — Add technical indicators
4. **Model Training** — Build and train an LSTM neural network
5. **Evaluation** — Measure accuracy on held-out test data
6. **Prediction** — Forecast future prices

##  Setup & Data Collection

In [104]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.data_fetcher import Fetcher
from src.utils import (
    add_indicators, scale_features,
    make_sequences, split_data,
    inverse_scale
)
from src.model import make_model, train, forecast, save_model

In [105]:
COIN = 'BTC'
PERIOD = '1DAY'       
HISTORY_DAYS = 365    
LOOKBACK = 60         
PREDICTION_DAYS = 7   

print(f'Configuration: {COIN} \n {PERIOD} \n {HISTORY_DAYS} days history \n {LOOKBACK} lookback \n {PREDICTION_DAYS} days prediction')

Configuration: BTC 
 1DAY 
 365 days history 
 60 lookback 
 7 days prediction


In [106]:
# Fetch data from CoinGecko
fetcher = Fetcher()
df = fetcher.get_history(COIN, period=PERIOD, days=HISTORY_DAYS)

print(f'\nFetched {len(df)} records for {COIN}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
df.head(10)

[CACHE] Using cached data: BTC_1DAY_365d.csv

Fetched 366 records for BTC
Date range: 2025-04-14 00:00:00 → 2026-04-13 08:20:57


,date,open,high,low,close,volume,source
0,2025-04-14,85297.000000,85732.000000,83197.000000,83600.820101,2.873113e+10,CoinGecko
1,2025-04-15,84523.452491,84523.452491,84523.452491,84523.452491,3.315558e+10,CoinGecko
2,2025-04-16,83656.492489,83656.492489,83656.492489,83656.492489,2.508467e+10,CoinGecko
3,2025-04-17,84105.779422,84105.779422,84105.779422,84105.779422,2.694811e+10,CoinGecko
4,2025-04-18,83637.000000,86186.000000,83219.000000,84930.908576,1.932382e+10,CoinGecko
5,2025-04-19,84433.750172,84433.750172,84433.750172,84433.750172,9.550691e+09,CoinGecko
6,2025-04-20,85126.662443,85126.662443,85126.662443,85126.662443,1.263598e+10,CoinGecko
7,2025-04-21,85073.165449,85073.165449,85073.165449,85073.165449,1.262584e+10,CoinGecko
8,2025-04-22,84940.000000,88260.000000,84038.000000,87452.046991,4.007003e+10,CoinGecko
9,2025-04-23,93576.165886,93576.165886,93576.165886,93576.165886,5.384199e+10,CoinGecko


##  Exploratory Data Analysis

In [107]:
df[['open', 'high', 'low', 'close', 'volume']].describe()

,open,high,low,close,volume
count,366.000000,366.000000,366.000000,366.000000,3.660000e+02
mean,97042.043712,97764.456280,96261.890706,97001.808911,4.581920e+10
std,17051.042201,17084.673268,17101.919126,17098.338184,2.245731e+10
min,62853.690384,62853.690384,60256.000000,62853.690384,8.960878e+09
25%,86843.750000,87550.787586,85691.000000,86877.026416,2.968806e+10
50%,101970.437951,102583.483788,101215.820147,101970.437951,4.304319e+10
75%,110961.632438,111605.321264,110211.846597,110961.632438,5.637118e+10
max,124740.000000,126080.000000,123560.993636,124773.508231,1.510022e+11


In [108]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3],
    subplot_titles=(f'{COIN}/USD — Price History', 'Volume')
)

fig.add_trace(
    go.Candlestick(
        x=df['date'], open=df['open'], high=df['high'],
        low=df['low'], close=df['close'], name='OHLC'
    ), row=1, col=1
)

fig.add_trace(
    go.Bar(x=df['date'], y=df['volume'], name='Volume',
           marker_color='rgba(100, 149, 237, 0.5)'),
    row=2, col=1
)

fig.update_layout(
    template='plotly_dark', height=700,
    title_text=f'{COIN} Historical Price & Volume',
    xaxis_rangeslider_visible=False
)
fig.show()

##  Feature Engineering

In [109]:
df_features = add_indicators(df)

df_features = df_features.dropna().reset_index(drop=True)

print(f'Features added: {[c for c in df_features.columns if c not in df.columns]}')
print(f'Final dataset shape: {df_features.shape}')
df_features.tail()

Features added: ['SMA_7', 'SMA_21', 'RSI_14', 'MACD', 'MACD_signal', 'BB_upper', 'BB_lower', 'price_change', 'volatility_7']
Final dataset shape: (366, 16)


,date,open,high,low,close,volume,source,SMA_7,SMA_21,RSI_14,MACD,MACD_signal,BB_upper,BB_lower,price_change,volatility_7
361,2026-04-10 00:00:00,71770.746850,71770.746850,71770.746850,71770.746850,3.929302e+10,CoinGecko,69565.306775,68768.297476,61.234495,205.090619,-327.496676,72611.732584,64746.429585,0.009191,0.019098
362,2026-04-11 00:00:00,72972.708236,72972.708236,72972.708236,72972.708236,3.835839e+10,CoinGecko,70427.166349,68883.539520,77.731630,467.121841,-168.572973,73267.917243,64514.160970,0.016747,0.018720
363,2026-04-12 00:00:00,73053.888085,73053.888085,73053.888085,73053.888085,2.386813e+10,CoinGecko,71248.543281,69089.270010,77.881260,673.569517,-0.144475,73872.677680,64429.901736,0.001112,0.019063
364,2026-04-13 00:00:00,71004.000000,73721.000000,70522.000000,70756.751055,3.010138e+10,CoinGecko,71501.575371,69227.740248,67.068891,644.392868,128.762994,73855.687945,64433.283752,-0.031444,0.023933
365,2026-04-13 08:20:57,71004.000000,73721.000000,70522.000000,70700.390380,2.852607e+10,CoinGecko,71763.883350,69218.576541,64.987568,609.694153,224.949226,73875.970591,64430.588968,-0.000797,0.023898


In [110]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_features['date'], y=df_features['close'],
                         name='Close Price', line=dict(color='#00d4ff', width=2)))
fig.add_trace(go.Scatter(x=df_features['date'], y=df_features['SMA_7'],
                         name='SMA 7', line=dict(color='#ff6b6b', width=1, dash='dash')))
fig.add_trace(go.Scatter(x=df_features['date'], y=df_features['SMA_21'],
                         name='SMA 21', line=dict(color='#ffd93d', width=1, dash='dash')))
fig.add_trace(go.Scatter(x=df_features['date'], y=df_features['BB_upper'],
                         name='BB Upper', line=dict(color='rgba(255,255,255,0.2)', width=1)))
fig.add_trace(go.Scatter(x=df_features['date'], y=df_features['BB_lower'],
                         name='BB Lower', line=dict(color='rgba(255,255,255,0.2)', width=1),
                         fill='tonexty', fillcolor='rgba(255,255,255,0.05)'))

fig.update_layout(template='plotly_dark', height=500,
                  title=f'{COIN} Close Price with Technical Indicators')
fig.show()

##  Data Preparation & Train/Test Split

In [111]:
scaled_features, target_scaler, feature_scaler, feature_cols = scale_features(df_features)

print(f'Features used ({len(feature_cols)}): {feature_cols}')
print(f'Scaled data shape: {scaled_features.shape}')

X, y = make_sequences(scaled_features, lookback=LOOKBACK)
print(f'\nSequences: X={X.shape}, y={y.shape}')

X_train, X_test, y_train, y_test = split_data(X, y, test_ratio=0.2)
print(f'Train: X={X_train.shape}, y={y_train.shape}')
print(f'Test:  X={X_test.shape}, y={y_test.shape}')

Features used (14): ['close', 'open', 'high', 'low', 'volume', 'SMA_7', 'SMA_21', 'RSI_14', 'MACD', 'MACD_signal', 'BB_upper', 'BB_lower', 'price_change', 'volatility_7']
Scaled data shape: (366, 14)

Sequences: X=(306, 60, 14), y=(306,)
Train: X=(244, 60, 14), y=(244,)
Test:  X=(62, 60, 14), y=(62,)


##  Model Training

In [112]:
input_shape = (X_train.shape[1], X_train.shape[2])
model = make_model(input_shape)
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_12 (LSTM)                  │ (None, 60, 128)        │        73,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 124,737 (487.25 KB)

 Trainable params: 124,737 (487.25 KB)

 Non-trainable params: 0 (0.00 B)

In [113]:
history = train(
    model, X_train, y_train,
    X_val=X_test, y_val=y_test,
    epochs=50, batch_size=32
)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - loss: 0.1884 - mae: 0.3284 - val_loss: 0.0476 - val_mae: 0.2101 - learning_rate: 0.0010
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0371 - mae: 0.1580 - val_loss: 0.0279 - val_mae: 0.1550 - learning_rate: 0.0010
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0232 - mae: 0.1204 - val_loss: 0.0082 - val_mae: 0.0757 - learning_rate: 0.0010
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0141 - mae: 0.0930 - val_loss: 0.0031 - val_mae: 0.0429 - learning_rate: 0.0010
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0131 - mae: 0.0899 - val_loss: 0.0019 - val_mae: 0.0369 - learning_rate: 0.0010
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.0102 - mae: 0.0815 - val_loss: 0.0021 - val_mae: 0.0389 - learning_rate: 0.0010
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0094 - mae: 0.0752 - val_loss: 0.0021 - val_mae: 0.0389 - learning_rate: 0.0010
Epoch 8/50
8/8 ━━━━━━━━━━━━

In [114]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss (MSE)', 'MAE'))

fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss',
                         line=dict(color='#00d4ff')), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Val Loss',
                         line=dict(color='#ff6b6b')), row=1, col=1)

fig.add_trace(go.Scatter(y=history.history['mae'], name='Train MAE',
                         line=dict(color='#00d4ff')), row=1, col=2)
fig.add_trace(go.Scatter(y=history.history['val_mae'], name='Val MAE',
                         line=dict(color='#ff6b6b')), row=1, col=2)

fig.update_layout(template='plotly_dark', height=400,
                  title_text='Training History')
fig.show()

##  Evaluation — Actual vs Predicted

In [115]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_pred_scaled = model.predict(X_test).flatten()

y_test_actual = inverse_scale(y_test, target_scaler)
y_pred_actual = inverse_scale(y_pred_scaled, target_scaler)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = np.mean(np.abs((y_test_actual - y_pred_actual) / y_test_actual)) * 100

print(f'Test Set Performance:')
print(f'RMSE:  ${rmse:,.2f}')
print(f'MAE:   ${mae:,.2f}')
print(f'MAPE:  {mape:.2f}%')

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 267ms/step
Test Set Performance:
RMSE:  $2,710.07
MAE:   $2,283.36
MAPE:  3.29%


In [116]:
test_dates = df_features['date'].iloc[-len(y_test_actual):].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dates, y=y_test_actual,
                         name='Actual', line=dict(color='#00d4ff', width=2)))
fig.add_trace(go.Scatter(x=test_dates, y=y_pred_actual,
                         name='Predicted', line=dict(color='#ff6b6b', width=2, dash='dash')))

fig.update_layout(
    template='plotly_dark', height=500,
    title=f'{COIN} — Actual vs Predicted (Test Set)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    annotations=[dict(
        text=f'RMSE: ${rmse:,.2f} | MAE: ${mae:,.2f} | MAPE: {mape:.2f}%',
        xref='paper', yref='paper', x=0.5, y=1.05,
        showarrow=False, font=dict(size=12, color='#aaa')
    )]
)
fig.show()

##  Save Model

In [117]:
model_path = save_model(model, COIN)
print(f'\nModel saved to: {model_path}')

import pickle
scalers_path = f'../models/{COIN.upper()}_scalers.pkl'
with open(scalers_path, 'wb') as f:
    pickle.dump({
        'target_scaler': target_scaler,
        'feature_scaler': feature_scaler,
        'feature_columns': feature_cols,
        'lookback': LOOKBACK
    }, f)
print(f'Scalers saved to: {scalers_path}')

[SAVED] Model saved: D:\Data_Science\Cryptoapp_2\backend\models\BTC_lstm.keras

Model saved to: D:\Data_Science\Cryptoapp_2\backend\models\BTC_lstm.keras
Scalers saved to: ../models/BTC_scalers.pkl


##  Future Price Prediction

In [118]:
last_sequence = scaled_features[-LOOKBACK:]
n_features = len(feature_cols)

future_prices = forecast(
    model, last_sequence, target_scaler, feature_scaler,
    n_features=n_features, days=PREDICTION_DAYS
)

last_date = df_features['date'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=PREDICTION_DAYS)

print(f'\n{COIN} Price Predictions ({PREDICTION_DAYS} days):')
print('=' * 40)
for d, p in zip(future_dates, future_prices):
    print(f'{d.strftime("%Y-%m-%d")}: ${p:,.2f}')


BTC Price Predictions (7 days):
2026-04-14: $70,121.61
2026-04-15: $70,487.04
2026-04-16: $70,820.40
2026-04-17: $71,114.32
2026-04-18: $71,366.41
2026-04-19: $71,577.49
2026-04-20: $71,750.42


In [120]:
fig = go.Figure()

recent = df_features.tail(90)
fig.add_trace(go.Scatter(
    x=recent['date'], y=recent['close'],
    name='Historical', line=dict(color='#00d4ff', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=[recent['date'].iloc[-1], future_dates[0]],
    y=[recent['close'].iloc[-1], future_prices[0]],
    name='Transition', line=dict(color='#888', width=1, dash='dot'),
    showlegend=False, mode='lines'
))

fig.add_trace(go.Scatter(
    x=future_dates, y=future_prices,
    name='Predicted', line=dict(color='#ff6b6b', width=3),
    mode='lines+markers',
    marker=dict(size=8, symbol='diamond', color='#ff6b6b')
))

fig.update_layout(
    template='plotly_dark', height=600,
    title=f'{COIN}/USD — Historical + Predicted Prices',
    xaxis_title='Date', yaxis_title='Price (USD)',
    legend=dict(x=0.01, y=0.99)
)
fig.show()